# 구글 코랩에 연결해서 진행


In [1]:
import sys, torch
print(sys.executable)
assert torch.cuda.is_available(), "GPU 미사용!"
print(torch.cuda.get_device_name(0))

C:\projects\PycharmProjects\PythonProject\RunModel\.venv\Scripts\python.exe
NVIDIA GeForce RTX 2060


In [2]:
from datasets import load_dataset
from dotenv import load_dotenv
import os


load_dotenv()
dataset = load_dataset("Cheukting/math-meta-reasoning-cleaned", token=os.getenv("HUGGINGFACEHUB_API_TOKEN"))
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'token_count'],
        num_rows: 987485
    })
})

In [3]:
from transformers import GPT2Tokenizer


tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(examples):
   return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [4]:
tokenized_datasets_split = tokenized_datasets["train"].shard(num_shards=100, index=0).train_test_split(test_size=0.2, shuffle=True)
tokenized_datasets_split

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'token_count', 'input_ids', 'attention_mask'],
        num_rows: 7900
    })
    test: Dataset({
        features: ['id', 'text', 'token_count', 'input_ids', 'attention_mask'],
        num_rows: 1975
    })
})

In [5]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results4',
    num_train_epochs=3,                  # 3 → 1 (학습용이면 충분)
    per_device_train_batch_size=4,       # 4 → 8 (GPU 활용률↑) <- 8로 늘리면 OOM 발생
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # 유효 배치는 16으로 유지
    warmup_steps=100,
    weight_decay=0.01,
    save_steps=300,
    save_total_limit=2,                  # ← 디스크 보호, 아래 설명
    logging_steps=100,
    fp16=True,                           # ← 가장 큰 효과
    # dataloader_pin_memory=False,       # ← CUDA에선 삭제 (기본값 True가 맞음)
)

In [6]:
from transformers import GPT2LMHeadModel, Trainer, DataCollatorForLanguageModeling

model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_datasets_split['train'],
   eval_dataset=tokenized_datasets_split['test'],
   data_collator=data_collator,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
trainer.train(resume_from_checkpoint=False)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.109516


In [ ]:
trainer.evaluate(tokenized_datasets_split['test'])

In [ ]:
trainer.save_model("./trained_model4")

In [ ]:
model.push_to_hub("gpt2-math-reasoning", private=True)

In [ ]:
# tokenizer.push_to_hub("gpt2-math-reasoning", private=True)